# Feature Store Approach


To organize the variables used in the model, a **Feature Store**-based approach was adopted. The goal is to separate **feature construction** from the modeling stage, allowing the same variables to be used consistently in both training and prediction.

Feature construction:

Features are created using **SQL queries**, stored in separate *.sql* files. This organization keeps the logic of each feature group isolated and makes maintenance and reuse easier.

The structure follows, for example:

```text
feature_store/
├── fs_cadastral.sql
├── fs_temporal.sql
├── fs_historico_financeiro.sql
├── fs_renda.sql
├── fs_funcionarios.sql
└── fs_historico_pagamentos.sql
```

Queries are parameterized by reference period. This way, the same query can be executed for different months, generating the features corresponding to each DATA_REF.

Feature Store ingestion:

Query execution is centralized in an **ingestion notebook**. This notebook reads each *.sql* file, runs the query for the defined periods, and writes the results to the respective Feature Store tables.

The flow can be represented as follows:

```text
Raw data
     ↓
SQL queries
     ↓
Feature construction
     ↓
Ingestion notebook
     ↓
Feature Store
     ↓
Training Set / Prediction
```

On the first run, if the table does not yet exist, it is created defining:

* *ID_CLIENTE*
* *ID_DOCUMENTO*
* *DATA_REF*

as feature keys;

* *DATA_REF* as the partition column.

On subsequent runs, new periods are added using **merge**, allowing the Feature Store to be updated without recreating the entire table.

Usage in training:

During training, the different feature tables are retrieved through the *FeatureEngineeringClient* and joined to the main dataset using the defined keys.

This allows building a **single training set**, combining the registration, temporal, and historical information required by the model.

An important advantage of this approach is ensuring that feature construction is **reproducible and organized**, while also making it easier to use the same features later in the prediction process.

> **Important:** because this is a temporal problem, historical features must be built using only information available up to the respective *DATA_REF*. This prevents future information from being used during training and reduces the risk of *data leakage*.



Features were divided into different groups according to their origin and purpose:

* **Cadastral**: customer registration characteristics;
* **Temporal**: information related to the reference period;
* **Financial history**: features related to financial behavior;
* **Income**: information and variations related to income;
* **Employees**: history and behavior of the number of employees;
* **Payment history**: metrics related to payment behavior.


# Feature Ideas


## Cadastral Feature Store


**Key:** ID_CLIENTE

Features related to registration characteristics and the customer's relationship with the company.

**Features**

* **Customer region:** geographic region obtained from registration information.
* **Relationship duration:** time elapsed between the customer's registration date and the reference date.


## Income Feature Store


**Key:** ID_CLIENTE, SAFRA_REF

Features intended to represent the **level, behavior, trend, and stability of customer income over time**.

**Income Aggregations**

Income statistics are calculated considering different temporal windows.

3 months

* Average income.
* Sum of income.
* Minimum income.
* Maximum income.

6 months

* Average income.
* Sum of income.
* Minimum income.
* Maximum income.

12 months

* Average income.
* Sum of income.
* Minimum income.
* Maximum income.

Lifetime

* Historical average income.
* Historical sum of income.
* Historical minimum income.
* Historical maximum income.

**Trends**

These aim to identify how customer income is evolving over time, comparing earlier periods with more recent ones.

* Income growth over 3 months.
* Income growth over 6 months.
* Income growth over 12 months.
* Income decline over 3 months.
* Income decline over 6 months.
* Income decline over 12 months.

These measures are calculated in a **temporal** way, using only periods before SAFRA_REF.

**Variability**

These aim to measure customer income **stability**.

* Standard deviation of income over 3 months.
* Standard deviation of income over 6 months.
* Standard deviation of income over 12 months.
* Coefficient of variation of income.

**History**

This aims to identify prolonged periods of income decline, representing possible changes in the customer's financial behavior.

* **Consecutive months of decline:** number of consecutive months in which income decreased relative to the previous period.

> The windows and metrics presented represent initial hypotheses. During development, they may be adjusted, removed, or complemented according to data availability and analysis results.


## Employees Feature Store


**Key:** ID_CLIENTE, SAFRA_REF

The employees Feature Store aims to represent the **size, evolution, and stability of the customer's employee headcount over time**. This information can help the model identify changes in company structure related to default risk.

**Features**

Current headcount

* Current number of employees.

Headcount growth

Calculates the evolution of the number of employees over different temporal windows:

* Headcount growth over the last 3 months.
* Headcount growth over the last 6 months.
* Headcount growth over the last 12 months.
* Headcount growth over the entire available history.

Headcount reduction

Identifies reductions in the number of employees:

* Headcount reduction over the last 3 months.
* Headcount reduction over the last 6 months.
* Headcount reduction over the last 12 months.
* Headcount reduction over the entire available history.

Historical statistics

Aims to identify relevant variations in employee headcount:

* Largest monthly growth observed.
* Largest monthly decline observed.

Comparison with company size

Compares the customer's current number of employees with the expected behavior for companies of the same size:

* Difference between the current number of employees and the average number of employees for the respective size category.

Efficiency

Relates headcount size to company income:

* **Income per employee:** monthly income divided by the number of employees.

> **Note:** these are the initially proposed features. During exploration and modeling, new variables may be created, some may be modified, and others may be discarded if they show low relevance, availability issues, or risk of *data leakage*.


## Payment History Feature Store


**Key:** ID_CLIENTE, SAFRA_REF

The payment history Feature Store aims to represent the **financial behavior and payment punctuality of the customer over time**. Variables are built from billing and payment history using different temporal windows.

**Features**

Delays

Aims to identify the frequency and intensity of customer delays:

* Delay flag over the last 3 months.
* Delay flag over the last 6 months.
* Delay flag over the last 12 months.
* Delay flag over the entire history.
* Number of delays over the last 3 months.
* Number of delays over the last 6 months.
* Number of delays over the last 12 months.
* Number of delays over the entire history.
* Days since the last delay.
* Average days of delay.
* Maximum number of days of delay.

Payments

Characterizes the customer's payment behavior:

* Days since the last payment.
* Number of early payments.
* Average days of early payment.
* Number of payments made on the due date.

Billing charges

Represents the volume and values of billing charges:

* Number of billing charges.
* Average billing charge amount.
* Largest billing charge amount.
* Total amount paid over the last 3 months.
* Total amount paid over the last 6 months.
* Total amount paid over the last 12 months.

On-time payment

Measures the customer's behavior relative to payment deadlines:

* Percentage of on-time payments.

Dates

Characterizes intervals related to the billing and payment cycle:

* Average time between issue and due date.
* Minimum time between issue and due date.
* Maximum time between issue and due date.
* Days remaining until due date.
* Average days between issue and payment.

Billing

Relates billing charge amount to the available payment period:

* Billing charge amount per day.

> **Note:** these are the initially proposed features. The final definition may change during exploration and modeling, including creation of new variables, removal of low-relevance variables, and adjustments to temporal windows. Features must also respect information availability at prediction time, avoiding *data leakage*.


## Temporal Feature Store


**Key:** ID_CLIENTE, SAFRA_REF

The temporal features Feature Store aims to represent the **customer's relationship with time**, using information available at prediction time. These variables help the model capture aspects related to relationship duration and the billing cycle.

Features

Relationship

* Days since customer registration.

Billing

* Days until due date.
* Time between issue and due date.

Payment history

* Days since the last payment.
* Days since the last delay.

> **Note:** these are the initially proposed features. The final definition may change during exploration and modeling, according to information availability, predictive relevance, and the need to avoid *data leakage*.
